# Cell 1 — Install and imports

In [1]:
!pip -q install ultralytics pyyaml

import os
import json
import shutil
from pathlib import Path
from collections import Counter

import torch
import pandas as pd
import numpy as np
from ultralytics import YOLO

print("Imports done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.1 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Imports done


# Cell 2 — Input from Notebook 1 

In [2]:
# Your saved Notebook 1 dataset
PREP_ROOT = "/kaggle/input/datasets/naresh26032005/vinbigdata-prep"
YOLO_INPUT_ROOT = os.path.join(PREP_ROOT, "yolo640")
SPLIT_DIR = os.path.join(PREP_ROOT, "splits")

assert os.path.exists(PREP_ROOT), f"PREP_ROOT not found: {PREP_ROOT}"
assert os.path.exists(YOLO_INPUT_ROOT), f"YOLO folder not found: {YOLO_INPUT_ROOT}"
assert os.path.exists(os.path.join(SPLIT_DIR, "train_split.csv")), "train_split.csv missing"
assert os.path.exists(os.path.join(SPLIT_DIR, "val_split.csv")), "val_split.csv missing"

print("PREP_ROOT:", PREP_ROOT)
print("YOLO_INPUT_ROOT:", YOLO_INPUT_ROOT)
print("SPLIT_DIR:", SPLIT_DIR)

PREP_ROOT: /kaggle/input/datasets/naresh26032005/vinbigdata-prep
YOLO_INPUT_ROOT: /kaggle/input/datasets/naresh26032005/vinbigdata-prep/yolo640
SPLIT_DIR: /kaggle/input/datasets/naresh26032005/vinbigdata-prep/splits


# Cell 3 — Load prep metadata

In [3]:
train_df = pd.read_csv(os.path.join(SPLIT_DIR, "train_split.csv"))
val_df = pd.read_csv(os.path.join(SPLIT_DIR, "val_split.csv"))

print("Train rows:", len(train_df))
print("Val rows:", len(val_df))
print(train_df.head())

NO_FINDING_ID = 14
YOLO_SIZE = 640

def parse_classes(s):
    if pd.isna(s) or str(s).strip() == "":
        return []
    return [int(x) for x in str(s).split() if str(x).strip().isdigit()]

def build_class_frequency(df):
    freq = Counter()
    for cls_str in df["classes"].tolist():
        cls_list = parse_classes(cls_str)
        for c in cls_list:
            if c != NO_FINDING_ID:
                freq[c] += 1
    return freq

train_freq = build_class_frequency(train_df)
print("Train class frequency:", train_freq)

def build_weighted_manifest(df, split_name, out_txt, repeat_cap=4):
    """
    Creates a weighted image manifest without duplicating images on disk.
    Rare abnormal classes get repeated more often.
    """
    os.makedirs(os.path.dirname(out_txt), exist_ok=True)

    # baseline for rarity weighting
    abn_freq_values = [v for k, v in train_freq.items() if k != NO_FINDING_ID and v > 0]
    base = float(np.median(abn_freq_values)) if len(abn_freq_values) > 0 else 1.0

    lines = []

    for _, row in df.iterrows():
        img_id = row["image_id"]
        classes = parse_classes(row["classes"])

        img_path = os.path.join(PREP_ROOT, "yolo640", "images", split_name, f"{img_id}.jpg")
        if not os.path.exists(img_path):
            continue

        # normal images stay once
        abnormal_classes = [c for c in classes if c != NO_FINDING_ID]
        if len(abnormal_classes) == 0:
            repeat = 1
        else:
            rarity = 0.0
            for c in abnormal_classes:
                rarity += base / max(train_freq.get(c, 1), 1)
            repeat = int(np.clip(round(rarity), 1, repeat_cap))

        for _ in range(repeat):
            lines.append(img_path)

    with open(out_txt, "w") as f:
        f.write("\n".join(lines))

    return out_txt, len(lines)

WORK_DIR = "/kaggle/working/yolo640_fixed"
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR, ignore_errors=True)
os.makedirs(WORK_DIR, exist_ok=True)

train_txt, train_count = build_weighted_manifest(
    train_df, "train", os.path.join(WORK_DIR, "train.txt"), repeat_cap=4
)
val_txt, val_count = build_weighted_manifest(
    val_df, "val", os.path.join(WORK_DIR, "val.txt"), repeat_cap=1
)

print("Train manifest:", train_txt, "lines:", train_count)
print("Val manifest:", val_txt, "lines:", val_count)

print("\nSample train paths:")
with open(train_txt, "r") as f:
    for _ in range(5):
        print(next(f).strip())

Train rows: 12000
Val rows: 3000
                           image_id  \
0  000434271f63a053c4128a0ba6352c7f   
1  0005e8e3701dfb1dd93d53e2ff537b6e   
2  0006e0a85696f6bb578e84fafa9a5607   
3  0007d316f756b3fa0baea2ff514ce945   
4  000ae00eb3942d27e0b97903dd563a6e   

                                        src_path_rel  \
0  /kaggle/input/competitions/vinbigdata-chest-xr...   
1  /kaggle/input/competitions/vinbigdata-chest-xr...   
2  /kaggle/input/competitions/vinbigdata-chest-xr...   
3  /kaggle/input/competitions/vinbigdata-chest-xr...   
4  /kaggle/input/competitions/vinbigdata-chest-xr...   

                                   cache_path_rel  has_abnormal  \
0  cache_640/000434271f63a053c4128a0ba6352c7f.jpg             0   
1  cache_640/0005e8e3701dfb1dd93d53e2ff537b6e.jpg             1   
2  cache_640/0006e0a85696f6bb578e84fafa9a5607.jpg             0   
3  cache_640/0007d316f756b3fa0baea2ff514ce945.jpg             1   
4  cache_640/000ae00eb3942d27e0b97903dd563a6e.jpg           

# Create a fixed YOLO YAML that points to the new manifests

In [4]:
CLASS_NAMES = [
    "Aortic enlargement",
    "Atelectasis",
    "Calcification",
    "Cardiomegaly",
    "Consolidation",
    "ILD",
    "Infiltration",
    "Lung Opacity",
    "Nodule/Mass",
    "Other lesion",
    "Pleural effusion",
    "Pleural thickening",
    "Pneumothorax",
    "Pulmonary fibrosis",
]

YOLO_YAML_FIXED = os.path.join(WORK_DIR, "vinbigdata_640_fixed.yaml")

yaml_text = f"""path: {PREP_ROOT}
train: {train_txt}
val: {val_txt}
names:
"""
for i, name in enumerate(CLASS_NAMES):
    yaml_text += f"  {i}: {name}\n"

with open(YOLO_YAML_FIXED, "w") as f:
    f.write(yaml_text)

print(open(YOLO_YAML_FIXED).read())

path: /kaggle/input/datasets/naresh26032005/vinbigdata-prep
train: /kaggle/working/yolo640_fixed/train.txt
val: /kaggle/working/yolo640_fixed/val.txt
names:
  0: Aortic enlargement
  1: Atelectasis
  2: Calcification
  3: Cardiomegaly
  4: Consolidation
  5: ILD
  6: Infiltration
  7: Lung Opacity
  8: Nodule/Mass
  9: Other lesion
  10: Pleural effusion
  11: Pleural thickening
  12: Pneumothorax
  13: Pulmonary fibrosis



# Cell 4 — Training configuration

In [5]:
DEVICE = 0 if torch.cuda.is_available() else "cpu"

MODEL_NAME = "yolov8m.pt"
EPOCHS = 25
IMGSZ = 640
BATCH = 16
WORKERS = 2

print("Device:", DEVICE)
print("Model:", MODEL_NAME)
print("Epochs:", EPOCHS)
print("Image size:", IMGSZ)
print("Batch:", BATCH)

Device: 0
Model: yolov8m.pt
Epochs: 25
Image size: 640
Batch: 16


# Cell 5 — Train YOLOv8 640

In [6]:
model = YOLO(MODEL_NAME)

model.train(
    data=YOLO_YAML_FIXED,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,

    # storage/runtime friendly
    cache=False,
    amp=True,
    pretrained=True,

    # optimizer / schedule
    optimizer="AdamW",
    lr0=1e-3,
    weight_decay=1e-4,
    cos_lr=True,
    warmup_epochs=2.0,
    patience=8,

    # safe medical augmentation
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
    degrees=5.0,
    translate=0.05,
    scale=0.10,
    fliplr=0.5,
    mosaic=0.45,
    mixup=0.05,
    close_mosaic=5,

    # reduce extra output size
    plots=True,
    save=True,

    project=WORK_DIR,
    name="yolov8m_640",
    verbose=True,
)

print("Training finished")

Ultralytics 8.4.41 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/yolo640_fixed/vinbigdata_640_fixed.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.05, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=0.45, multi_scale=0.0, name=yolov8m_640, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, ove

# Cell 6 — Validate the best model

In [7]:
best_weights = os.path.join(WORK_DIR, "yolov8m_640", "weights", "best.pt")
last_weights = os.path.join(WORK_DIR, "yolov8m_640", "weights", "last.pt")

print("Best weights:", best_weights)
print("Last weights:", last_weights)

assert os.path.exists(best_weights), "best.pt not found after training"

trained_model = YOLO(best_weights)

print("\nRunning validation...")
metrics = trained_model.val(
    data=YOLO_YAML_FIXED,
    imgsz=IMGSZ,
    device=DEVICE
)

print(metrics)

Best weights: /kaggle/working/yolo640_fixed/yolov8m_640/weights/best.pt
Last weights: /kaggle/working/yolo640_fixed/yolov8m_640/weights/last.pt

Running validation...
Ultralytics 8.4.41 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,847,866 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 108.5±46.1 MB/s, size: 66.8 KB)
val: Scanning /kaggle/input/datasets/naresh26032005/vinbigdata-prep/yolo640/labels/val... 3000 images, 2121 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3000/3000 642.3it/s 4.7s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/naresh26032005/vinbigdata-prep/yolo640/labels is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 188/188 2.4it/s 1:17
                   all       3000       4739      0.328      0.302      0.261       0.12
    Aortic enlargement        638        716      0.697   

# Cell 7 — Copy best model to a simple path

In [8]:
final_model_path = os.path.join(WORK_DIR, "best_yolov8m_640.pt")

shutil.copy2(best_weights, final_model_path)
print("Copied best model to:", final_model_path)
print("Model size (MB):", round(os.path.getsize(final_model_path) / (1024 * 1024), 2))

Copied best model to: /kaggle/working/yolo640_fixed/best_yolov8m_640.pt
Model size (MB): 49.63


# Cell 8 — Save run summary

In [9]:
summary = {
    "prep_root": PREP_ROOT,
    "yolo_yaml": YOLO_YAML_FIXED,
    "train_txt": train_txt,
    "val_txt": val_txt,
    "best_weights": best_weights,
    "last_weights": last_weights,
    "final_model_path": final_model_path,
    "model_name": MODEL_NAME,
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "workers": WORKERS,
}

with open(os.path.join(WORK_DIR, "train_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

{
  "prep_root": "/kaggle/input/datasets/naresh26032005/vinbigdata-prep",
  "yolo_yaml": "/kaggle/working/yolo640_fixed/vinbigdata_640_fixed.yaml",
  "train_txt": "/kaggle/working/yolo640_fixed/train.txt",
  "val_txt": "/kaggle/working/yolo640_fixed/val.txt",
  "best_weights": "/kaggle/working/yolo640_fixed/yolov8m_640/weights/best.pt",
  "last_weights": "/kaggle/working/yolo640_fixed/yolov8m_640/weights/last.pt",
  "final_model_path": "/kaggle/working/yolo640_fixed/best_yolov8m_640.pt",
  "model_name": "yolov8m.pt",
  "epochs": 25,
  "imgsz": 640,
  "batch": 16,
  "workers": 2
}


# Cell 9 — Check output size

In [10]:
total_bytes = 0
for root, _, files in os.walk(WORK_DIR):
    for f in files:
        total_bytes += os.path.getsize(os.path.join(root, f))

print(f"Approx YOLO output size: {total_bytes / (1024**3):.3f} GB")

Approx YOLO output size: 0.157 GB


# Cell 10 — Final folder check

In [11]:
print("YOLO training output:")
for root, dirs, files in os.walk(WORK_DIR):
    level = root.replace(WORK_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = "  " * (level + 1)
    for f in files[:5]:
        print(f"{subindent}{f}")

YOLO training output:
yolo640_fixed/
  val.txt
  train_summary.json
  best_yolov8m_640.pt
  vinbigdata_640_fixed.yaml
  train.txt
  yolov8m_640/
    confusion_matrix.png
    BoxR_curve.png
    train_batch2.jpg
    train_batch0.jpg
    results.csv
    weights/
      best.pt
      last.pt


In [12]:
import os
from IPython.display import FileLink

# 1. Zip your folder (the fastest compression method)
# Replace 'your_folder_name' with the actual folder you want to download
!zip -r -q output_data.zip /kaggle/working/

# 2. Generate and display the fast download link
FileLink(r'output_data.zip')

/kaggle/working/output_data.zip